In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# Train BEV v4 Cleaned + DINOv2 ViT-Base

Self-contained notebook for `v4` training with:
- merged `train + val` pool;
- empty-GT removal;
- smart deduplication by near-static consecutive frames;
- test-matched validation split targeting about 200 samples;
- rover embeddings with `Other=0`, top-25 frequent train rovers getting unique IDs;
- `DINOv2 ViT-Base` image backbone loaded through `torch.hub`;
- safe train loop with only `val@0.5` and `ema val@0.5` during training.

This notebook does not modify the original dataset structure. All caches, hub downloads, and checkpoints are written only to `RUN_DIR`.



In [ ]:
!ls /kaggle

In [ ]:
import os, copy, time, json, math, random, gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

DATA_TRAIN = Path('/kaggle/input/datasets/letiti6e/bev-yandex/autonomy_yandex_dataset_train_kaggle_safe_v2/autonomy_yandex_dataset_train/')
DATA_VAL   = Path('/kaggle/input/datasets/letiti6e/bev-yandex/autonomy_yandex_dataset_val_kaggle_safe/autonomy_yandex_dataset_val/')
DATA_TEST  = Path('/kaggle/input/datasets/letiti6e/bev-yandex/autonomy_yandex_dataset_test_kaggle_safe/autonomy_yandex_dataset_test/')

cfg = {
    'run_dir': './runs/v4_dinov2_cleaned',
    'img_hw': (392, 700),
    'batch_size': 4,
    'val_batch_size': 1,
    'grad_accum': 4,
    'epochs': 20,
    'warmup_epochs': 3,
    'freeze_backbone_epochs': 2,
    'lr_backbone': 3e-5,
    'lr_head': 3e-4,
    'weight_decay': 1e-4,
    'embed_weight_decay': 1e-2,
    'pos_weight': 5.0,
    'num_workers': 4,
    'val_num_workers': 0,
    'seed': 42,
    'ema_decay': 0.995,
    'val_target_size': 200,
    'min_rover_count': 30,
    'topk_rovers': 25,
    'rover_emb_dim': 8,
    'rover_cond_dim': 8,
    'mae_dedup_thr': 0.02,
    'dedup_camera': '/camera/inner/frontal/middle',
    'use_clean_cache': True,
    'backbone_name': 'dinov2_vitb14',
    'hub_repo': 'facebookresearch/dinov2',
    'backbone_out_dim': 768,
    'backbone_proj_dim': 128,
}

RUN_DIR = Path(cfg['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = RUN_DIR / 'preproc_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TORCH_HUB_DIR = RUN_DIR / 'torch_hub'
TORCH_HUB_DIR.mkdir(parents=True, exist_ok=True)
torch.hub.set_dir(str(TORCH_HUB_DIR))

random.seed(cfg['seed'])
np.random.seed(cfg['seed'])
torch.manual_seed(cfg['seed'])

if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('device:', device)
if device.type == 'cuda':
    print('cuda devices:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))
print('torch hub dir:', TORCH_HUB_DIR)
print('img_hw:', cfg['img_hw'])
print('train batch_size:', cfg['batch_size'], '| train workers:', cfg['num_workers'])
print('val batch_size:', cfg['val_batch_size'], '| val workers:', cfg['val_num_workers'])

with open(RUN_DIR / 'config.json', 'w') as f:
    json.dump({k: str(v) for k, v in cfg.items()}, f, indent=2)



In [ ]:
CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)
Z_LEVELS = (0.3, 1.0, 2.0, 3.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def resolve_info_path(base_dir: Path, p) -> Path:
    p = Path(p)
    if p.is_absolute() and p.exists():
        return p
    if p.exists():
        return p
    cand = base_dir / p
    if cand.exists():
        return cand
    cand = base_dir.parent / p
    if cand.exists():
        return cand
    return base_dir.parent / p


def load_info_with_root(data_dir: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(data_dir / 'info.csv', index_col=0).reset_index(drop=True).copy()
    df['__data_root'] = str(data_dir)
    df['__source_split'] = split_name
    return df


def resolve_row_path(row: pd.Series, key: str) -> Path:
    return resolve_info_path(Path(row['__data_root']), row[key])


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.cleaning import build_img_hash, clean_merged_info, compute_gt_stats, smart_deduplicate


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.splits import build_rover_vocab_from_train, encode_rover, make_test_matched_split_target


In [ ]:
clean_info, dedup_report, clean_summary = clean_merged_info(
    DATA_TRAIN,
    DATA_VAL,
    cache_dir=CACHE_DIR,
    mae_thr=cfg['mae_dedup_thr'],
    dedup_camera=cfg['dedup_camera'],
    use_cache=cfg['use_clean_cache'],
)
print(json.dumps(clean_summary, indent=2, ensure_ascii=False))
display(clean_info.head(3))
if len(dedup_report):
    display(dedup_report.head(10))

train_idx, val_idx = make_test_matched_split_target(
    clean_info,
    DATA_TEST / 'info.csv',
    target_val_size=cfg['val_target_size'],
    seed=cfg['seed'],
    cache_path=CACHE_DIR / 'test_matched_val200_split.npz',
)
print('train_idx:', len(train_idx), 'val_idx:', len(val_idx))

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=cfg['min_rover_count'],
    topk=cfg['topk_rovers'],
)
rover_stats.to_csv(RUN_DIR / 'rover_embedding_stats.csv', index=False)
print('num rover classes including Other:', len(rover_vocab))
print('top rover mapping:', rover_vocab)
display(rover_stats.head(30))


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.data import BEVDatasetV4Clean


In [ ]:
class _UNetBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class SmallUNet(nn.Module):
    def __init__(self, in_c, base_c=32, out_c=1):
        super().__init__()
        self.in_proj = nn.Conv2d(in_c, base_c, 1)
        c = [base_c, base_c * 2, base_c * 4, base_c * 8]
        self.enc1 = _UNetBlock(c[0], c[0])
        self.enc2 = _UNetBlock(c[0], c[1])
        self.enc3 = _UNetBlock(c[1], c[2])
        self.bot = _UNetBlock(c[2], c[3])
        self.dec3 = _UNetBlock(c[3] + c[2], c[2])
        self.dec2 = _UNetBlock(c[2] + c[1], c[1])
        self.dec1 = _UNetBlock(c[1] + c[0], c[0])
        self.out = nn.Conv2d(c[0], out_c, 1)
        self.pool = nn.MaxPool2d(2)

    def _up(self, x, ref):
        return F.interpolate(x, size=ref.shape[-2:], mode='bilinear', align_corners=False)

    def forward(self, x):
        x = self.in_proj(x)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bot(self.pool(e3))
        d3 = self.dec3(torch.cat([self._up(b, e3), e3], 1))
        d2 = self.dec2(torch.cat([self._up(d3, e2), e2], 1))
        d1 = self.dec1(torch.cat([self._up(d2, e1), e1], 1))
        return self.out(d1)


class _DINOv2ViTBackbone(nn.Module):
    def __init__(self,
                 hub_repo: str = 'facebookresearch/dinov2',
                 backbone_name: str = 'dinov2_vitb14',
                 out_dim: int = 768,
                 proj_dim: int = 128,
                 patch_size: int = 14):
        super().__init__()
        self.hub_repo = hub_repo
        self.backbone_name = backbone_name
        self.out_dim = out_dim
        self.patch_size = patch_size
        self.vit = self._load_hub_model()
        self.proj = nn.Conv2d(out_dim, proj_dim, 1)

    def _load_hub_model(self):
        last_err = None
        attempts = [
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, pretrained=True),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, source='github'),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, source='github', pretrained=True),
        ]
        for kwargs in attempts:
            try:
                return torch.hub.load(**kwargs)
            except Exception as e:
                last_err = e
        raise RuntimeError(
            f'Failed to load DINOv2 backbone {self.backbone_name} from {self.hub_repo}. '
            f'Last error: {last_err}'
        )

    def _extract_feature_map(self, x: torch.Tensor) -> torch.Tensor:
        B, _, H, W = x.shape
        Hp = H // self.patch_size
        Wp = W // self.patch_size
        expected_tokens = Hp * Wp

        if hasattr(self.vit, 'get_intermediate_layers'):
            try:
                inter = self.vit.get_intermediate_layers(x, n=1, reshape=True, return_class_token=False)
                feat = inter[0] if isinstance(inter, (list, tuple)) else inter
                if isinstance(feat, (list, tuple)):
                    feat = feat[0]
                if feat.ndim == 4:
                    return feat
            except Exception:
                pass

        tokens = None
        if hasattr(self.vit, 'forward_features'):
            feats = self.vit.forward_features(x)
            if isinstance(feats, dict):
                if 'x_norm_patchtokens' in feats:
                    tokens = feats['x_norm_patchtokens']
                elif 'patch_tokens' in feats:
                    tokens = feats['patch_tokens']
                elif 'x_prenorm' in feats:
                    tokens = feats['x_prenorm']
            else:
                tokens = feats
        if tokens is None:
            tokens = self.vit(x)

        if isinstance(tokens, (list, tuple)):
            tokens = tokens[0]
        if not isinstance(tokens, torch.Tensor):
            raise RuntimeError(f'Unexpected DINO output type: {type(tokens)}')
        if tokens.ndim != 3:
            raise RuntimeError(f'Unexpected DINO token shape: {tuple(tokens.shape)}')

        if tokens.shape[1] == expected_tokens + 1:
            tokens = tokens[:, 1:, :]
        elif tokens.shape[1] != expected_tokens:
            raise RuntimeError(
                f'Unexpected number of DINO tokens: got {tokens.shape[1]}, expected {expected_tokens} '
                f'for img_hw={tuple(x.shape[-2:])}'
            )

        feat = tokens.transpose(1, 2).reshape(B, self.out_dim, Hp, Wp).contiguous()
        return feat

    def forward(self, x):
        feat = self._extract_feature_map(x)
        feat = self.proj(feat)
        return feat


class MultiCamBEVv4DINOv2Clean(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 hub_repo: str = 'facebookresearch/dinov2',
                 backbone_name: str = 'dinov2_vitb14',
                 backbone_out_dim: int = 768,
                 backbone_proj_dim: int = 128):
        super().__init__()
        self.n_cameras = n_cameras
        self.bev_h, self.bev_w = BEV_H, BEV_W
        self.x_range, self.y_range = X_RANGE, Y_RANGE
        self.z_levels = Z_LEVELS
        self.rover_cond_dim = rover_cond_dim

        self.backbone = _DINOv2ViTBackbone(
            hub_repo=hub_repo,
            backbone_name=backbone_name,
            out_dim=backbone_out_dim,
            proj_dim=backbone_proj_dim,
            patch_size=14,
        )
        if freeze_backbone:
            for p in self.backbone.vit.parameters():
                p.requires_grad = False

        self.feat_proj = nn.Conv2d(backbone_proj_dim, 64, 1)
        self.register_buffer('ego_voxels', self._make_ego_voxels(), persistent=False)

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        in_c = 64 * len(self.z_levels) + rover_cond_dim
        self.bev_decoder = SmallUNet(in_c=in_c, base_c=32, out_c=1)

    def _make_ego_voxels(self):
        xs = torch.linspace(self.x_range[0] + BEV_RES / 2, self.x_range[1] - BEV_RES / 2, self.bev_h)
        ys = torch.linspace(self.y_range[0] + BEV_RES / 2, self.y_range[1] - BEV_RES / 2, self.bev_w)
        zs = torch.tensor(self.z_levels, dtype=torch.float32)
        Z, X, Y = torch.meshgrid(zs, xs, ys, indexing='ij')
        ones = torch.ones_like(X)
        return torch.stack([X, Y, Z, ones], dim=-1)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C_, Hi, Wi = images.shape
        assert N == self.n_cameras

        feat = self.backbone(images.reshape(B * N, C_, Hi, Wi))
        feat = self.feat_proj(feat)
        Hf, Wf = feat.shape[-2:]
        feat = feat.reshape(B, N, 64, Hf, Wf)

        Z, H, W, _ = self.ego_voxels.shape
        V = Z * H * W
        voxels = self.ego_voxels.reshape(-1, 4).unsqueeze(0).unsqueeze(0).expand(B, N, -1, -1)

        p_cam = torch.einsum('bnij,bnvj->bniv', car2cams, voxels)
        p_cam_3d = p_cam[:, :, :3]
        uv_homog = torch.einsum('bnij,bnjv->bniv', intrinsics, p_cam_3d)
        z = uv_homog[:, :, 2]
        u = uv_homog[:, :, 0] / z.clamp(min=1e-3)
        v = uv_homog[:, :, 1] / z.clamp(min=1e-3)

        u_n = 2.0 * u / Wi - 1.0
        v_n = 2.0 * v / Hi - 1.0
        valid = (z > 0.1) & (u_n.abs() <= 1.0) & (v_n.abs() <= 1.0)

        grid = torch.stack([u_n, v_n], dim=-1).reshape(B * N, V, 1, 2)
        sampled = F.grid_sample(
            feat.reshape(B * N, 64, Hf, Wf),
            grid,
            mode='bilinear',
            padding_mode='zeros',
            align_corners=False,
        )
        sampled = sampled.squeeze(-1).reshape(B, N, 64, V)

        valid_f = valid.float().unsqueeze(2)
        sampled = sampled * valid_f
        agg = sampled.sum(dim=1) / valid_f.sum(dim=1).clamp(min=1.0)
        agg = agg.reshape(B, 64, Z, H, W).reshape(B, 64 * Z, H, W)

        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, H, W)
        agg = torch.cat([agg, rover_map], dim=1)
        return self.bev_decoder(agg)



In [ ]:
def _lovasz_grad(gt_sorted: torch.Tensor) -> torch.Tensor:
    p = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1.0 - gt_sorted).float().cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1]
    return jaccard


def lovasz_hinge_flat(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    if labels.numel() == 0:
        return logits.sum() * 0.0
    signs = 2.0 * labels.float() - 1.0
    errors = 1.0 - logits * signs
    errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
    gt_sorted = labels[perm]
    grad = _lovasz_grad(gt_sorted)
    return torch.dot(F.relu(errors_sorted), grad)


class CompoundLossV2(nn.Module):
    def __init__(self, pos_weight: float = 5.0,
                 weight_bce: float = 0.5,
                 weight_dice: float = 0.3,
                 weight_lovasz: float = 0.2,
                 ignore_value: int = 255):
        super().__init__()
        self.register_buffer('pos_weight', torch.tensor([pos_weight]))
        self.weight_bce = weight_bce
        self.weight_dice = weight_dice
        self.weight_lovasz = weight_lovasz
        self.ignore_value = ignore_value

    def forward(self, logits: torch.Tensor, gt: torch.Tensor):
        valid = (gt != self.ignore_value)
        gt_f = (gt == 1).float()

        bce_per = F.binary_cross_entropy_with_logits(logits, gt_f, pos_weight=self.pos_weight, reduction='none')
        bce = (bce_per * valid.float()).sum() / valid.float().sum().clamp(min=1.0)

        prob = torch.sigmoid(logits) * valid.float()
        gt_d = gt_f * valid.float()
        inter = (prob * gt_d).sum(dim=(1, 2, 3))
        denom = prob.sum(dim=(1, 2, 3)) + gt_d.sum(dim=(1, 2, 3))
        dice = (1.0 - (2 * inter + 1) / (denom + 1)).mean()

        lov_logits = logits[valid]
        lov_gt = gt_f[valid]
        lov = lovasz_hinge_flat(lov_logits, lov_gt) if lov_logits.numel() > 0 else logits.sum() * 0.0

        total = self.weight_bce * bce + self.weight_dice * dice + self.weight_lovasz * lov
        parts = {'bce': bce.item(), 'dice': dice.item(), 'lovasz': lov.item()}
        return total, parts


@torch.no_grad()
def iou_binary_batch(logits: torch.Tensor, gt: torch.Tensor, threshold: float = 0.5, ignore_value: int = 255):
    probs = torch.sigmoid(logits)
    pred = probs > threshold
    valid = gt != ignore_value
    gt_b = (gt == 1) & valid
    pred = pred & valid
    inter = (pred & gt_b).sum().item()
    union = (pred | gt_b).sum().item()
    return inter, union


@torch.no_grad()
def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


@torch.inference_mode()
def evaluate_iou(model, loader, threshold=0.5, desc='val@0.5'):
    model.eval()
    inter = 0
    union = 0
    pbar = tqdm(loader, desc=desc, leave=False)
    for batch in pbar:
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id).float()
        i, u = iou_binary_batch(logits, gt, threshold=threshold)
        inter += i
        union += u
        pbar.set_postfix(iou=f'{inter / max(union, 1):.4f}')
    return inter / max(union, 1)


@torch.no_grad()
def streaming_threshold_sweep(model, loader, thresholds):
    model.eval()
    inter = torch.zeros(len(thresholds), dtype=torch.float64)
    union = torch.zeros(len(thresholds), dtype=torch.float64)
    for batch in tqdm(loader, desc='threshold sweep'):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt']
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id).float()
        probs = torch.sigmoid(logits).cpu()
        valid = gt != 255
        gt_b = ((gt == 1) & valid).float()
        for ti, t in enumerate(thresholds):
            pred = ((probs > t) & valid).float()
            inter[ti] += (pred * gt_b).sum().item()
            union[ti] += ((pred + gt_b).clamp(0, 1)).sum().item()
    return {float(t): float(inter[i] / max(union[i].item(), 1.0)) for i, t in enumerate(thresholds)}



In [ ]:
cfg['batch_size']

In [ ]:
ds_train_warm = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
ds_train_aug = BEVDatasetV4Clean(train_info, mode='train', img_hw=cfg['img_hw'], aug=True, rover_vocab=rover_vocab)
ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

loader_warm = DataLoader(
    ds_train_warm,
    batch_size=cfg['batch_size'],
    shuffle=True,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_train = DataLoader(
    ds_train_aug,
    batch_size=cfg['batch_size'],
    shuffle=True,
    num_workers=cfg['num_workers'],
    pin_memory=(device.type == 'cuda'),
    drop_last=True,
)
loader_val = DataLoader(
    ds_val,
    batch_size=cfg['val_batch_size'],
    shuffle=False,
    num_workers=cfg['val_num_workers'],
    pin_memory=(device.type == 'cuda'),
)

print('loader_warm batch_size:', loader_warm.batch_size, '| workers:', loader_warm.num_workers)
print('loader_train batch_size:', loader_train.batch_size, '| workers:', loader_train.num_workers)
print('loader_val batch_size:', loader_val.batch_size, '| workers:', loader_val.num_workers)

sample = ds_train_warm[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v), v)



In [ ]:
model = MultiCamBEVv4DINOv2Clean(
    num_rover_classes=len(rover_vocab),
    rover_emb_dim=cfg['rover_emb_dim'],
    rover_cond_dim=cfg['rover_cond_dim'],
    freeze_backbone=False,
    hub_repo=cfg['hub_repo'],
    backbone_name=cfg['backbone_name'],
    backbone_out_dim=cfg['backbone_out_dim'],
    backbone_proj_dim=cfg['backbone_proj_dim'],
).to(device)


def set_backbone_trainable(model, trainable: bool):
    for p in model.backbone.vit.parameters():
        p.requires_grad = trainable


set_backbone_trainable(model, False)

backbone_params = []
embed_params = []
other_params = []
for name, p in model.named_parameters():
    if name.startswith('backbone.vit.'):
        backbone_params.append(p)
    elif name.startswith('rover_embed.'):
        embed_params.append(p)
    else:
        other_params.append(p)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['lr_backbone'], 'weight_decay': cfg['weight_decay']},
    {'params': other_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['weight_decay']},
    {'params': embed_params, 'lr': cfg['lr_head'], 'weight_decay': cfg['embed_weight_decay']},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
loss_fn = CompoundLossV2(pos_weight=cfg['pos_weight']).to(device)
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

ema_model = copy.deepcopy(model).eval()
for p in ema_model.parameters():
    p.requires_grad = False

@torch.no_grad()
def update_ema(ema_model, model, decay):
    ema_params = dict(ema_model.named_parameters())
    model_params = dict(model.named_parameters())
    for name, param in model_params.items():
        ema_params[name].mul_(decay).add_(param.data, alpha=1.0 - decay)
    ema_buffers = dict(ema_model.named_buffers())
    model_buffers = dict(model.named_buffers())
    for name, buf in model_buffers.items():
        ema_buffers[name].copy_(buf)

print('params M:', sum(p.numel() for p in model.parameters()) / 1e6)
print('backbone frozen at start:', not any(p.requires_grad for p in model.backbone.vit.parameters()))

with torch.no_grad():
    batch = next(iter(loader_warm))
    images = batch['images'].to(device)
    intr = batch['intrinsics'].to(device)
    c2c = batch['car2cams'].to(device)
    rover_id = batch['rover_id'].to(device)

    feat = model.backbone(images.reshape(-1, images.shape[2], images.shape[3], images.shape[4]))
    out = model(images, intr, c2c, rover_id)

    patch_h = cfg['img_hw'][0] // 14
    patch_w = cfg['img_hw'][1] // 14
    print('expected patch grid:', (patch_h, patch_w))
    print('backbone feature shape:', tuple(feat.shape))
    print('sanity output shape:', tuple(out.shape))

cleanup_cuda()



In [ ]:
log = []
best_iou = -1.0
best_ema_iou = -1.0
start = time.time()
backbone_unfrozen = False

for epoch in range(cfg['epochs']):
    if (not backbone_unfrozen) and epoch >= cfg['freeze_backbone_epochs']:
        set_backbone_trainable(model, True)
        backbone_unfrozen = True
        print(f'unfroze DINO backbone at epoch {epoch:02d}')

    model.train()
    loader = loader_warm if epoch < cfg['warmup_epochs'] else loader_train
    phase = 'WARM' if epoch < cfg['warmup_epochs'] else 'AUG'

    losses = []
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc=f'ep{epoch:02d} [{phase}]')
    for step, batch in enumerate(pbar):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id)
            loss, parts = loss_fn(logits, gt)
            loss = loss / cfg['grad_accum']

        scaler.scale(loss).backward()
        if (step + 1) % cfg['grad_accum'] == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_model, model, cfg['ema_decay'])

        losses.append(loss.item() * cfg['grad_accum'])
        if step % 20 == 0:
            pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.3f}', bce=f"{parts['bce']:.2f}")

    if len(loader) % cfg['grad_accum'] != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        update_ema(ema_model, model, cfg['ema_decay'])

    scheduler.step()

    cleanup_cuda()
    print('start val model @0.5')
    val_iou_05 = evaluate_iou(model, loader_val, threshold=0.5, desc=f'val model ep{epoch:02d}')

    cleanup_cuda()
    print('start val ema @0.5')
    val_iou_05_ema = evaluate_iou(ema_model, loader_val, threshold=0.5, desc=f'val ema ep{epoch:02d}')
    cleanup_cuda()

    elapsed = (time.time() - start) / 60
    backbone_grad_enabled = any(p.requires_grad for p in model.backbone.vit.parameters())

    row = {
        'epoch': epoch,
        'loss': float(np.mean(losses)),
        'val_iou_05': float(val_iou_05),
        'val_iou_05_ema': float(val_iou_05_ema),
        'backbone_trainable': bool(backbone_grad_enabled),
        'minutes': float(elapsed),
    }

    print(
        f"ep{epoch:02d} | loss {np.mean(losses):.3f} | "
        f"val@0.5 {val_iou_05:.4f} | ema@0.5 {val_iou_05_ema:.4f} | "
        f"backbone_trainable={backbone_grad_enabled} | {elapsed:.1f}m"
    )

    if val_iou_05 > best_iou:
        best_iou = val_iou_05
        torch.save({
            'model_type': 'v4_dinov2_cleaned',
            'model': model.state_dict(),
            'epoch': epoch,
            'best_iou': float(val_iou_05),
            'best_t': 0.5,
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'best.pt')
        print('  new best model:', val_iou_05)

    if val_iou_05_ema > best_ema_iou:
        best_ema_iou = val_iou_05_ema
        torch.save({
            'model_type': 'v4_dinov2_cleaned',
            'model': model.state_dict(),
            'ema': ema_model.state_dict(),
            'epoch': epoch,
            'best_ema_iou': float(val_iou_05_ema),
            'best_t_ema': 0.5,
            'rover_vocab': rover_vocab,
            'cfg': cfg,
        }, RUN_DIR / 'ema_best.pt')
        print('  new best ema:', val_iou_05_ema)

    log.append(row)
    pd.DataFrame(log).to_csv(RUN_DIR / 'log.csv', index=False)
    torch.save({
        'model_type': 'v4_dinov2_cleaned',
        'model': model.state_dict(),
        'ema': ema_model.state_dict(),
        'epoch': epoch,
        'rover_vocab': rover_vocab,
        'cfg': cfg,
    }, RUN_DIR / 'last.pt')




In [2]:
!kaggle kernels output letiti6e/notebook140df2ea97 -p .

Output file downloaded to ./runs/v4_dinov2_cleaned/best.pt
Output file downloaded to ./runs/v4_dinov2_cleaned/config.json
Output file downloaded to ./runs/v4_dinov2_cleaned/ema_best.pt
Output file downloaded to ./runs/v4_dinov2_cleaned/last.pt
Output file downloaded to ./runs/v4_dinov2_cleaned/log.csv
Output file downloaded to ./runs/v4_dinov2_cleaned/preproc_cache/clean_summary.json
Output file downloaded to ./runs/v4_dinov2_cleaned/preproc_cache/dedup_report.csv
Output file downloaded to ./runs/v4_dinov2_cleaned/preproc_cache/merged_cleaned.csv
Output file downloaded to ./runs/v4_dinov2_cleaned/preproc_cache/merged_gt_stats.csv
Output file downloaded to ./runs/v4_dinov2_cleaned/preproc_cache/test_matched_val200_split.npz
Output file downloaded to ./runs/v4_dinov2_cleaned/rover_embedding_stats.csv
Output file downloaded to ./runs/v4_dinov2_cleaned/torch_hub/checkpoints/dinov2_vitb14_pretrain.pth
Output file downloaded to ./runs/v4_dinov2_cleaned/torch_hub/facebookresearch_dinov2_main/

## Notes

- `clean_merged_info(...)` only writes caches and reports into `RUN_DIR / preproc_cache`.
- The original dataset folders are never modified.
- The `rover_vocab` is built strictly from the cleaned training split, so rare rovers and unseen rovers map to `Other=0`.
- Validation is selected from the merged cleaned pool and targets about 200 samples while staying group-aware by `(rover, ride_date)`.
- DINOv2 weights are expected to come from `torch.hub`, and the notebook will fail early on the sanity-check cell if the backbone cannot be loaded.



In [3]:

import os
import gc
import json
import math
import hashlib
import zipfile
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'


In [4]:
DATA_TRAIN = Path('/kaggle/input/datasets/letiti6e/bev-yandex/autonomy_yandex_dataset_train_kaggle_safe_v2/autonomy_yandex_dataset_train/')
DATA_VAL   = Path('/kaggle/input/datasets/letiti6e/bev-yandex/autonomy_yandex_dataset_val_kaggle_safe/autonomy_yandex_dataset_val/')
DATA_TEST  = Path('/kaggle/input/datasets/letiti6e/bev-yandex/autonomy_yandex_dataset_test_kaggle_safe/autonomy_yandex_dataset_test/')

In [7]:

RUN_DIR = Path('/kaggle/working/runs/v4_dinov2_cleaned')
assert RUN_DIR.exists(), RUN_DIR

BEST_PT = RUN_DIR / 'best.pt'
EMA_BEST_PT = RUN_DIR / 'ema_best.pt'
LAST_PT = RUN_DIR / 'last.pt'
CFG_JSON = RUN_DIR / 'config.json'
CACHE_DIR = RUN_DIR / 'preproc_cache'
TORCH_HUB_DIR = RUN_DIR / 'torch_hub'

assert BEST_PT.exists(), BEST_PT
assert EMA_BEST_PT.exists(), EMA_BEST_PT
assert CACHE_DIR.exists(), CACHE_DIR
assert TORCH_HUB_DIR.exists(), TORCH_HUB_DIR

torch.hub.set_dir(str(TORCH_HUB_DIR))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('run_dir:', RUN_DIR)
print('torch_hub_dir:', TORCH_HUB_DIR)


device: cuda
run_dir: /kaggle/working/runs/v4_dinov2_cleaned
torch_hub_dir: /kaggle/working/runs/v4_dinov2_cleaned/torch_hub


In [8]:
# Берем cfg из чекпоинта, чтобы не гадать
probe_ckpt = torch.load(EMA_BEST_PT if EMA_BEST_PT.exists() else BEST_PT, map_location='cpu')
ckpt_cfg = probe_ckpt.get('cfg', {})
print('ckpt cfg keys:', sorted(ckpt_cfg.keys()))

cfg = {
    'img_hw': tuple(ckpt_cfg.get('img_hw', (392, 700))),
    'rover_emb_dim': int(ckpt_cfg.get('rover_emb_dim', 8)),
    'rover_cond_dim': int(ckpt_cfg.get('rover_cond_dim', 8)),
    'backbone_name': ckpt_cfg.get('backbone_name', 'dinov2_vitb14'),
    'hub_repo': ckpt_cfg.get('hub_repo', 'facebookresearch/dinov2'),
    'backbone_out_dim': int(ckpt_cfg.get('backbone_out_dim', 768)),
    'backbone_proj_dim': int(ckpt_cfg.get('backbone_proj_dim', 128)),
    'val_batch_size': 1,
    'test_batch_size': 2,
    'val_num_workers': 0,
    'test_num_workers': 2,
}

cfg


ckpt cfg keys: ['backbone_name', 'backbone_out_dim', 'backbone_proj_dim', 'batch_size', 'dedup_camera', 'ema_decay', 'embed_weight_decay', 'epochs', 'freeze_backbone_epochs', 'grad_accum', 'hub_repo', 'img_hw', 'lr_backbone', 'lr_head', 'mae_dedup_thr', 'min_rover_count', 'num_workers', 'pos_weight', 'rover_cond_dim', 'rover_emb_dim', 'run_dir', 'seed', 'topk_rovers', 'use_clean_cache', 'val_batch_size', 'val_num_workers', 'val_target_size', 'warmup_epochs', 'weight_decay']


{'img_hw': (392, 700),
 'rover_emb_dim': 8,
 'rover_cond_dim': 8,
 'backbone_name': 'dinov2_vitb14',
 'hub_repo': 'facebookresearch/dinov2',
 'backbone_out_dim': 768,
 'backbone_proj_dim': 128,
 'val_batch_size': 1,
 'test_batch_size': 2,
 'val_num_workers': 0,
 'test_num_workers': 2}

In [9]:
CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)
Z_LEVELS = (0.3, 1.0, 2.0, 3.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.geometry import load_info_with_root, resolve_info_path, resolve_row_path
from src.splits import build_rover_vocab_from_train, encode_rover
from src.utils.training import cleanup_cuda


In [11]:
# Используем уже готовый cleaned cache и split cache из output тренировки
merged_cleaned_csv = CACHE_DIR / 'merged_cleaned.csv'
split_npz = CACHE_DIR / 'test_matched_val200_split.npz'

assert merged_cleaned_csv.exists(), merged_cleaned_csv
assert split_npz.exists(), split_npz

clean_info = pd.read_csv(merged_cleaned_csv)
split = np.load(split_npz)
train_idx = split['train_idx'].tolist()
val_idx = split['val_idx'].tolist()

train_info = clean_info.iloc[train_idx].reset_index(drop=True).copy()
val_info = clean_info.iloc[val_idx].reset_index(drop=True).copy()
test_info = load_info_with_root(DATA_TEST, 'test')

rover_vocab, rover_stats = build_rover_vocab_from_train(
    train_info,
    min_count=int(ckpt_cfg.get('min_rover_count', 30)),
    topk=int(ckpt_cfg.get('topk_rovers', 25)),
)

print('train:', len(train_info), 'val:', len(val_info), 'test:', len(test_info))
print('num rover classes:', len(rover_vocab))
display(rover_stats.head(20))


train: 4273 val: 220 test: 2000
num rover classes: 26


,rover,count,embedding_id,bucket
0,orvy,638,1,unique
1,shelly,496,2,unique
2,lerita,327,3,unique
3,ward,239,4,unique
4,ravine,208,5,unique
5,greben,194,6,unique
6,lucky,187,7,unique
7,miro,120,8,unique
8,benzon,114,9,unique
9,natelio,108,10,unique


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.data import BEVDatasetV4Clean


In [13]:
class _UNetBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class SmallUNet(nn.Module):
    def __init__(self, in_c, base_c=32, out_c=1):
        super().__init__()
        self.in_proj = nn.Conv2d(in_c, base_c, 1)
        c = [base_c, base_c * 2, base_c * 4, base_c * 8]
        self.enc1 = _UNetBlock(c[0], c[0])
        self.enc2 = _UNetBlock(c[0], c[1])
        self.enc3 = _UNetBlock(c[1], c[2])
        self.bot = _UNetBlock(c[2], c[3])
        self.dec3 = _UNetBlock(c[3] + c[2], c[2])
        self.dec2 = _UNetBlock(c[2] + c[1], c[1])
        self.dec1 = _UNetBlock(c[1] + c[0], c[0])
        self.out = nn.Conv2d(c[0], out_c, 1)
        self.pool = nn.MaxPool2d(2)

    def _up(self, x, ref):
        return F.interpolate(x, size=ref.shape[-2:], mode='bilinear', align_corners=False)

    def forward(self, x):
        x = self.in_proj(x)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bot(self.pool(e3))
        d3 = self.dec3(torch.cat([self._up(b, e3), e3], 1))
        d2 = self.dec2(torch.cat([self._up(d3, e2), e2], 1))
        d1 = self.dec1(torch.cat([self._up(d2, e1), e1], 1))
        return self.out(d1)


class _DINOv2ViTBackbone(nn.Module):
    def __init__(self,
                 hub_repo: str = 'facebookresearch/dinov2',
                 backbone_name: str = 'dinov2_vitb14',
                 out_dim: int = 768,
                 proj_dim: int = 128,
                 patch_size: int = 14,
                 torch_hub_dir: Path | None = None):
        super().__init__()
        self.hub_repo = hub_repo
        self.backbone_name = backbone_name
        self.out_dim = out_dim
        self.patch_size = patch_size
        self.torch_hub_dir = Path(torch_hub_dir) if torch_hub_dir is not None else None
        self.vit = self._load_hub_model()
        self.proj = nn.Conv2d(out_dim, proj_dim, 1)

    def _load_hub_model(self):
        last_err = None
        attempts = []

        if self.torch_hub_dir is not None:
            local_repo = self.torch_hub_dir / 'facebookresearch_dinov2_main'
            if local_repo.exists():
                attempts.extend([
                    dict(repo_or_dir=str(local_repo), model=self.backbone_name, source='local'),
                    dict(repo_or_dir=str(local_repo), model=self.backbone_name, source='local', pretrained=True),
                ])

        attempts.extend([
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, pretrained=True),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, source='github'),
            dict(repo_or_dir=self.hub_repo, model=self.backbone_name, source='github', pretrained=True),
        ])

        for kwargs in attempts:
            try:
                return torch.hub.load(**kwargs)
            except Exception as e:
                last_err = e

        raise RuntimeError(
            f'Failed to load DINOv2 backbone {self.backbone_name}. Last error: {last_err}'
        )

    def _extract_feature_map(self, x: torch.Tensor) -> torch.Tensor:
        B, _, H, W = x.shape
        Hp = H // self.patch_size
        Wp = W // self.patch_size
        expected_tokens = Hp * Wp

        if hasattr(self.vit, 'get_intermediate_layers'):
            try:
                inter = self.vit.get_intermediate_layers(x, n=1, reshape=True, return_class_token=False)
                feat = inter[0] if isinstance(inter, (list, tuple)) else inter
                if isinstance(feat, (list, tuple)):
                    feat = feat[0]
                if feat.ndim == 4:
                    return feat
            except Exception:
                pass

        tokens = None
        if hasattr(self.vit, 'forward_features'):
            feats = self.vit.forward_features(x)
            if isinstance(feats, dict):
                if 'x_norm_patchtokens' in feats:
                    tokens = feats['x_norm_patchtokens']
                elif 'patch_tokens' in feats:
                    tokens = feats['patch_tokens']
                elif 'x_prenorm' in feats:
                    tokens = feats['x_prenorm']
            else:
                tokens = feats

        if tokens is None:
            tokens = self.vit(x)

        if isinstance(tokens, (list, tuple)):
            tokens = tokens[0]
        if not isinstance(tokens, torch.Tensor):
            raise RuntimeError(f'Unexpected DINO output type: {type(tokens)}')
        if tokens.ndim != 3:
            raise RuntimeError(f'Unexpected DINO token shape: {tuple(tokens.shape)}')

        if tokens.shape[1] == expected_tokens + 1:
            tokens = tokens[:, 1:, :]
        elif tokens.shape[1] != expected_tokens:
            raise RuntimeError(
                f'Unexpected number of DINO tokens: got {tokens.shape[1]}, expected {expected_tokens}'
            )

        feat = tokens.transpose(1, 2).reshape(B, self.out_dim, Hp, Wp).contiguous()
        return feat

    def forward(self, x):
        feat = self._extract_feature_map(x)
        feat = self.proj(feat)
        return feat


class MultiCamBEVv4DINOv2Clean(nn.Module):
    def __init__(self, num_rover_classes: int,
                 rover_emb_dim: int = 8,
                 rover_cond_dim: int = 8,
                 n_cameras: int = 4,
                 freeze_backbone: bool = False,
                 hub_repo: str = 'facebookresearch/dinov2',
                 backbone_name: str = 'dinov2_vitb14',
                 backbone_out_dim: int = 768,
                 backbone_proj_dim: int = 128,
                 torch_hub_dir: Path | None = None):
        super().__init__()
        self.n_cameras = n_cameras
        self.bev_h, self.bev_w = BEV_H, BEV_W
        self.x_range, self.y_range = X_RANGE, Y_RANGE
        self.z_levels = Z_LEVELS
        self.rover_cond_dim = rover_cond_dim

        self.backbone = _DINOv2ViTBackbone(
            hub_repo=hub_repo,
            backbone_name=backbone_name,
            out_dim=backbone_out_dim,
            proj_dim=backbone_proj_dim,
            patch_size=14,
            torch_hub_dir=torch_hub_dir,
        )
        if freeze_backbone:
            for p in self.backbone.vit.parameters():
                p.requires_grad = False

        self.feat_proj = nn.Conv2d(backbone_proj_dim, 64, 1)
        self.register_buffer('ego_voxels', self._make_ego_voxels(), persistent=False)

        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        nn.init.normal_(self.rover_embed.weight, std=0.02)
        self.rover_mlp = nn.Sequential(
            nn.Linear(rover_emb_dim, 16),
            nn.ReLU(inplace=True),
            nn.Linear(16, rover_cond_dim),
            nn.ReLU(inplace=True),
        )

        in_c = 64 * len(self.z_levels) + rover_cond_dim
        self.bev_decoder = SmallUNet(in_c=in_c, base_c=32, out_c=1)

    def _make_ego_voxels(self):
        xs = torch.linspace(self.x_range[0] + BEV_RES / 2, self.x_range[1] - BEV_RES / 2, self.bev_h)
        ys = torch.linspace(self.y_range[0] + BEV_RES / 2, self.y_range[1] - BEV_RES / 2, self.bev_w)
        zs = torch.tensor(self.z_levels, dtype=torch.float32)
        Z, X, Y = torch.meshgrid(zs, xs, ys, indexing='ij')
        ones = torch.ones_like(X)
        return torch.stack([X, Y, Z, ones], dim=-1)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C_, Hi, Wi = images.shape
        assert N == self.n_cameras

        feat = self.backbone(images.reshape(B * N, C_, Hi, Wi))
        feat = self.feat_proj(feat)
        Hf, Wf = feat.shape[-2:]
        feat = feat.reshape(B, N, 64, Hf, Wf)

        Z, H, W, _ = self.ego_voxels.shape
        V = Z * H * W
        voxels = self.ego_voxels.reshape(-1, 4).unsqueeze(0).unsqueeze(0).expand(B, N, -1, -1)

        p_cam = torch.einsum('bnij,bnvj->bniv', car2cams, voxels)
        p_cam_3d = p_cam[:, :, :3]
        uv_homog = torch.einsum('bnij,bnjv->bniv', intrinsics, p_cam_3d)
        z = uv_homog[:, :, 2]
        u = uv_homog[:, :, 0] / z.clamp(min=1e-3)
        v = uv_homog[:, :, 1] / z.clamp(min=1e-3)

        u_n = 2.0 * u / Wi - 1.0
        v_n = 2.0 * v / Hi - 1.0
        valid = (z > 0.1) & (u_n.abs() <= 1.0) & (v_n.abs() <= 1.0)

        grid = torch.stack([u_n, v_n], dim=-1).reshape(B * N, V, 1, 2)
        sampled = F.grid_sample(
            feat.reshape(B * N, 64, Hf, Wf),
            grid,
            mode='bilinear',
            padding_mode='zeros',
            align_corners=False,
        )
        sampled = sampled.squeeze(-1).reshape(B, N, 64, V)

        valid_f = valid.float().unsqueeze(2)
        sampled = sampled * valid_f
        agg = sampled.sum(dim=1) / valid_f.sum(dim=1).clamp(min=1.0)
        agg = agg.reshape(B, 64, Z, H, W).reshape(B, 64 * Z, H, W)

        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, H, W)
        agg = torch.cat([agg, rover_map], dim=1)
        return self.bev_decoder(agg)


In [14]:
@torch.no_grad()
def iou_binary_batch(logits: torch.Tensor, gt: torch.Tensor, threshold: float = 0.5, ignore_value: int = 255):
    probs = torch.sigmoid(logits)
    pred = probs > threshold
    valid = gt != ignore_value
    gt_b = (gt == 1) & valid
    pred = pred & valid
    inter = (pred & gt_b).sum().item()
    union = (pred | gt_b).sum().item()
    return inter, union


@torch.inference_mode()
def collect_val_probs(model, loader, cache_path: Path):
    if cache_path.exists():
        print('loading val probs from', cache_path)
        d = torch.load(cache_path, map_location='cpu')
        return d['probs'], d['gt']

    model.eval()
    probs_list = []
    gt_list = []
    for batch in tqdm(loader, desc=f'collect val probs -> {cache_path.name}'):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)
        gt = batch['gt'].cpu()

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id).float()
        probs = torch.sigmoid(logits).cpu().to(torch.float16)

        probs_list.append(probs)
        gt_list.append(gt)

    probs = torch.cat(probs_list, dim=0)
    gt = torch.cat(gt_list, dim=0)
    torch.save({'probs': probs, 'gt': gt}, cache_path)
    return probs, gt


def threshold_sweep_from_cached_probs(probs: torch.Tensor, gt: torch.Tensor, thresholds):
    inter = torch.zeros(len(thresholds), dtype=torch.float64)
    union = torch.zeros(len(thresholds), dtype=torch.float64)

    valid = gt != 255
    gt_b = ((gt == 1) & valid).float()

    for ti, t in enumerate(thresholds):
        pred = ((probs > t) & valid).float()
        inter[ti] = (pred * gt_b).sum().item()
        union[ti] = ((pred + gt_b).clamp(0, 1)).sum().item()

    return {
        float(t): float(inter[i] / max(union[i].item(), 1.0))
        for i, t in enumerate(thresholds)
    }


@torch.inference_mode()
def collect_test_probs(model, loader, cache_path: Path):
    if cache_path.exists():
        print('loading test probs from', cache_path)
        return torch.load(cache_path, map_location='cpu')

    model.eval()
    probs_list = []
    for batch in tqdm(loader, desc=f'collect test probs -> {cache_path.name}'):
        images = batch['images'].to(device, non_blocking=True)
        intr = batch['intrinsics'].to(device, non_blocking=True)
        c2c = batch['car2cams'].to(device, non_blocking=True)
        rover_id = batch['rover_id'].to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            logits = model(images, intr, c2c, rover_id).float()
        probs = torch.sigmoid(logits).cpu().to(torch.float16)
        probs_list.append(probs)

    probs = torch.cat(probs_list, dim=0)
    torch.save(probs, cache_path)
    return probs


In [15]:
ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)
ds_test = BEVDatasetV4Clean(test_info, mode='test', img_hw=cfg['img_hw'], aug=False, rover_vocab=rover_vocab)

loader_val = DataLoader(
    ds_val,
    batch_size=cfg['val_batch_size'],
    shuffle=False,
    num_workers=cfg['val_num_workers'],
    pin_memory=(device.type == 'cuda'),
)

loader_test = DataLoader(
    ds_test,
    batch_size=cfg['test_batch_size'],
    shuffle=False,
    num_workers=cfg['test_num_workers'],
    pin_memory=(device.type == 'cuda'),
)

print('val size:', len(ds_val), 'test size:', len(ds_test))


val size: 220 test size: 2000


In [16]:
def build_model():
    return MultiCamBEVv4DINOv2Clean(
        num_rover_classes=len(rover_vocab),
        rover_emb_dim=cfg['rover_emb_dim'],
        rover_cond_dim=cfg['rover_cond_dim'],
        freeze_backbone=False,
        hub_repo=cfg['hub_repo'],
        backbone_name=cfg['backbone_name'],
        backbone_out_dim=cfg['backbone_out_dim'],
        backbone_proj_dim=cfg['backbone_proj_dim'],
        torch_hub_dir=TORCH_HUB_DIR,
    ).to(device)


def load_candidate_model(candidate_name: str):
    if candidate_name == 'best':
        ckpt_path = BEST_PT
        ckpt = torch.load(ckpt_path, map_location='cpu')
        state = ckpt['model']
    elif candidate_name == 'ema_best':
        ckpt_path = EMA_BEST_PT
        ckpt = torch.load(ckpt_path, map_location='cpu')
        state = ckpt['ema'] if 'ema' in ckpt else ckpt['model']
    else:
        raise ValueError(candidate_name)

    model = build_model()
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(candidate_name, 'loaded from', ckpt_path)
    print('missing:', len(missing), 'unexpected:', len(unexpected))
    if len(missing):
        print('sample missing:', missing[:10])
    if len(unexpected):
        print('sample unexpected:', unexpected[:10])
    return model


In [17]:
thresholds = [round(x, 2) for x in np.arange(0.20, 0.82, 0.02)]
eval_dir = RUN_DIR / 'inference_eval'
eval_dir.mkdir(parents=True, exist_ok=True)

results = []

for candidate_name in ['best', 'ema_best']:
    cleanup_cuda()
    model = load_candidate_model(candidate_name)

    val_probs, val_gt = collect_val_probs(
        model,
        loader_val,
        cache_path=eval_dir / f'val_probs_{candidate_name}.pt',
    )

    iou_by_t = threshold_sweep_from_cached_probs(val_probs, val_gt, thresholds)
    best_t, best_iou = max(iou_by_t.items(), key=lambda kv: kv[1])

    df = pd.DataFrame({
        'threshold': list(iou_by_t.keys()),
        'iou': list(iou_by_t.values()),
    })
    df.to_csv(eval_dir / f'threshold_sweep_{candidate_name}.csv', index=False)

    print('\n', '=' * 100)
    print(candidate_name, 'best_t =', best_t, 'best_iou =', best_iou)
    display(df)

    results.append({
        'candidate': candidate_name,
        'best_t': float(best_t),
        'best_iou': float(best_iou),
    })

    del model
    cleanup_cuda()

results_df = pd.DataFrame(results).sort_values(
    ['best_iou', 'candidate'],
    ascending=[False, True],
).reset_index(drop=True)

display(results_df)


/kaggle/working/runs/v4_dinov2_cleaned/torch_hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/runs/v4_dinov2_cleaned/torch_hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/runs/v4_dinov2_cleaned/torch_hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


best loaded from /kaggle/working/runs/v4_dinov2_cleaned/best.pt
missing: 0 unexpected: 0


collect val probs -> val_probs_best.pt: 100%|██████████| 220/220 [01:21<00:00,  2.70it/s]



best best_t = 0.54 best_iou = 0.6022469474067883


,threshold,iou
0,0.20,0.569418
1,0.22,0.574456
2,0.24,0.578927
3,0.26,0.582810
4,0.28,0.586265
5,0.30,0.589332
6,0.32,0.591860
7,0.34,0.594020
8,0.36,0.595573
9,0.38,0.596918


ema_best loaded from /kaggle/working/runs/v4_dinov2_cleaned/ema_best.pt
missing: 0 unexpected: 0


collect val probs -> val_probs_ema_best.pt: 100%|██████████| 220/220 [00:45<00:00,  4.79it/s]



ema_best best_t = 0.58 best_iou = 0.5993952351959271


,threshold,iou
0,0.20,0.563853
1,0.22,0.568617
2,0.24,0.573070
3,0.26,0.576806
4,0.28,0.580277
5,0.30,0.583560
6,0.32,0.586698
7,0.34,0.589288
8,0.36,0.591529
9,0.38,0.593189


,candidate,best_t,best_iou
0,best,0.54,0.602247
1,ema_best,0.58,0.599395


In [18]:
# При равенстве предпочитаем ema_best
results_df['ema_priority'] = (results_df['candidate'] == 'ema_best').astype(int)
results_df = results_df.sort_values(
    ['best_iou', 'ema_priority'],
    ascending=[False, False],
).reset_index(drop=True)

selected_name = results_df.iloc[0]['candidate']
selected_threshold = float(results_df.iloc[0]['best_t'])
selected_iou = float(results_df.iloc[0]['best_iou'])

print('selected candidate:', selected_name)
print('selected threshold:', selected_threshold)
print('selected val iou:', selected_iou)


selected candidate: best
selected threshold: 0.54
selected val iou: 0.6022469474067883


In [19]:
cleanup_cuda()
model = load_candidate_model(selected_name)

test_probs = collect_test_probs(
    model,
    loader_test,
    cache_path=eval_dir / f'test_probs_{selected_name}.pt',
)

print('test_probs shape:', tuple(test_probs.shape))

del model
cleanup_cuda()


best loaded from /kaggle/working/runs/v4_dinov2_cleaned/best.pt
missing: 0 unexpected: 0


collect test probs -> test_probs_best.pt: 100%|██████████| 1000/1000 [04:39<00:00,  3.57it/s]


test_probs shape: (2000, 1, 188, 126)


In [21]:
cleanup_cuda()
model = load_candidate_model("ema_best")

test_probs_ema = collect_test_probs(
    model,
    loader_test,
    cache_path=eval_dir / f'test_probs_ema_best.pt',
)

print('test_probs shape:', tuple(test_probs.shape))

del model
cleanup_cuda()

ema_best loaded from /kaggle/working/runs/v4_dinov2_cleaned/ema_best.pt
missing: 0 unexpected: 0


collect test probs -> test_probs_ema_best.pt: 100%|██████████| 1000/1000 [03:33<00:00,  4.69it/s]


test_probs shape: (2000, 1, 188, 126)


In [22]:
def _pred_name_from_row(row):
    if 'predicted_occupancy_grid' in row:
        return Path(row['predicted_occupancy_grid']).name
    if 'predicted_static_grid' in row:
        return Path(row['predicted_static_grid']).name
    return f'{int(row.name)}.npy'


def make_submission_from_probs(test_probs: torch.Tensor, threshold: float, tag: str):
    work_dir = RUN_DIR / 'submissions'
    work_dir.mkdir(parents=True, exist_ok=True)

    thr_tag = f'{threshold:.2f}'.replace('.', 'p')
    pred_dir = work_dir / f'predicted_static_grids_{tag}_{thr_tag}'
    pred_dir.mkdir(parents=True, exist_ok=True)

    for p in pred_dir.glob('*.npy'):
        p.unlink()

    preds = (test_probs > threshold).numpy().astype(np.int32)
    assert len(preds) == len(test_info), (len(preds), len(test_info))

    for i, row in test_info.iterrows():
        out_name = _pred_name_from_row(row)
        np.save(pred_dir / out_name, preds[i].reshape(1, BEV_H, BEV_W))

    zip_path = work_dir / f'submission_{RUN_DIR.name}_{tag}_t_{thr_tag}.zip'
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        zf.write(DATA_TEST / 'info.csv', arcname='info.csv')
        for npy in sorted(pred_dir.glob('*.npy')):
            zf.write(npy, arcname=f'predicted_static_grids/{npy.name}')

    with zipfile.ZipFile(zip_path, 'r') as zf:
        bad = zf.testzip()
        assert bad is None, bad

    h = hashlib.sha256()
    with open(zip_path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)

    result = {
        'threshold': float(threshold),
        'zip_path': str(zip_path.resolve()),
        'size_mb': round(zip_path.stat().st_size / 1e6, 3),
        'sha256': h.hexdigest(),
    }
    print(json.dumps(result, indent=2, ensure_ascii=False))
    return result


In [25]:
thresholds = sorted(set([
    0.5,
    0.52,
    0.54,
    0.56,
    0.58,
    0.6,
    0.62,
    0.64,
    0.66,
    0.68,
    0.7,
    0.72,
    0.74,
    0.76,
    0.78,
    0.8,
    0.82,
    0.84,
    0.86,
    0.88
]))
thresholds = [t for t in thresholds if 0.05 <= t <= 0.95]

print('submit_thresholds:', thresholds)


submit_thresholds: [0.5, 0.52, 0.54, 0.56, 0.58, 0.6, 0.62, 0.64, 0.66, 0.68, 0.7, 0.72, 0.74, 0.76, 0.78, 0.8, 0.82, 0.84, 0.86, 0.88]


In [27]:
submission_results = []

tag = "best"

for t in thresholds:
    print('\n' + '=' * 100)
    print(f'building submission for threshold={t:.2f}')
    result = make_submission_from_probs(test_probs, t, tag=tag)
    submission_results.append(result)

submission_results_df = pd.DataFrame(submission_results)
display(submission_results_df)



building submission for threshold=0.50
{
  "threshold": 0.5,
  "zip_path": "/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p50.zip",
  "size_mb": 2.856,
  "sha256": "5ab64c8917be9c30d0b5d58b70fee2616b3dd68589ba0db91e71b35aebab203d"
}

building submission for threshold=0.52
{
  "threshold": 0.52,
  "zip_path": "/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p52.zip",
  "size_mb": 2.849,
  "sha256": "5609fc3132fa9b0e94d7f1dd7555c5a12fd102b4660963766208da67bfe873ca"
}

building submission for threshold=0.54
{
  "threshold": 0.54,
  "zip_path": "/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p54.zip",
  "size_mb": 2.849,
  "sha256": "9f88508e44cd522912e890d1d115dcc587f9fdbb354aca9545b08a9cdc522af5"
}

building submission for threshold=0.56
{
  "threshold": 0.56,
  "zip_path": "/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p56.

,threshold,zip_path,size_mb,sha256
0,0.50,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.856,5ab64c8917be9c30d0b5d58b70fee2616b3dd68589ba0d...
1,0.52,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.849,5609fc3132fa9b0e94d7f1dd7555c5a12fd102b4660963...
2,0.54,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.849,9f88508e44cd522912e890d1d115dcc587f9fdbb354aca...
3,0.56,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.850,7981744148cc25fb50002d5d886e731b417ee5a429ee5b...
4,0.58,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.850,b3becf3f54a913987d7dbbf4a1bf7414ce7f7dc3ab3234...
5,0.60,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.852,e3676165af5f5f8461913315bd989b87efe65c6eb6f597...
6,0.62,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.852,2f03cac46f762a83ed48f9495ddaf24c3de8f887bece4a...
7,0.64,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.852,8de4f23c5a6736431a8a459b7aa96000c9bfdbd375eaa6...
8,0.66,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.851,5a96594aeb6421d3591d45ca7745a88ded1e08e72227bc...
9,0.68,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.849,d7255921c19fd3838231ec67c3134311cccd8166420c73...


In [28]:
submission_results_ema = []

tag = "ema"

for t in thresholds:
    print('\n' + '=' * 100)
    print(f'building submission for threshold={t:.2f}')
    result = make_submission_from_probs(test_probs_ema, t, tag=tag)
    submission_results_ema.append(result)

submission_results_df_ema = pd.DataFrame(submission_results_ema)
display(submission_results_df_ema)



building submission for threshold=0.50
{
  "threshold": 0.5,
  "zip_path": "/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_ema_t_0p50.zip",
  "size_mb": 2.818,
  "sha256": "96a75e3fde2831ab903ec8e25da973564f0b0409853a39da4a21fbc21db159ec"
}

building submission for threshold=0.52
{
  "threshold": 0.52,
  "zip_path": "/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_ema_t_0p52.zip",
  "size_mb": 2.811,
  "sha256": "3ccc76a996c206c0ef1244ddfce407fcc1296e811dd17490a6f3dac1ac059418"
}

building submission for threshold=0.54
{
  "threshold": 0.54,
  "zip_path": "/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_ema_t_0p54.zip",
  "size_mb": 2.807,
  "sha256": "51e2c54298e6de8801a4b1148322d3e11d44458508d356d4198457496bfb5eab"
}

building submission for threshold=0.56
{
  "threshold": 0.56,
  "zip_path": "/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_ema_t_0p56.zip"

,threshold,zip_path,size_mb,sha256
0,0.50,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.818,96a75e3fde2831ab903ec8e25da973564f0b0409853a39...
1,0.52,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.811,3ccc76a996c206c0ef1244ddfce407fcc1296e811dd174...
2,0.54,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.807,51e2c54298e6de8801a4b1148322d3e11d44458508d356...
3,0.56,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.804,85f826358e035332c76392d26e6352f57b9f62fe2540cc...
4,0.58,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.805,a296843b7a78218e5a198fd5f8673bf618048324a1d14e...
5,0.60,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.803,9278b214c6e2d48723c5378053f9bfcee71552d12940cf...
6,0.62,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.803,93734597d6c339f0d83925134927ed70c65ec13971997d...
7,0.64,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.802,c60873e53eaaf2fd5149b8678feee19d73991616733e44...
8,0.66,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.802,6ccb3db0ff4d28aaf823ce2b137357660bb315c6d2d219...
9,0.68,/kaggle/working/runs/v4_dinov2_cleaned/submiss...,2.798,4ecbff501218531c59c747b238bf83e651e7f94f80021f...


In [29]:
submission_results + submission_results_ema

[{'threshold': 0.5,
  'zip_path': '/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p50.zip',
  'size_mb': 2.856,
  'sha256': '5ab64c8917be9c30d0b5d58b70fee2616b3dd68589ba0db91e71b35aebab203d'},
 {'threshold': 0.52,
  'zip_path': '/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p52.zip',
  'size_mb': 2.849,
  'sha256': '5609fc3132fa9b0e94d7f1dd7555c5a12fd102b4660963766208da67bfe873ca'},
 {'threshold': 0.54,
  'zip_path': '/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p54.zip',
  'size_mb': 2.849,
  'sha256': '9f88508e44cd522912e890d1d115dcc587f9fdbb354aca9545b08a9cdc522af5'},
 {'threshold': 0.56,
  'zip_path': '/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p56.zip',
  'size_mb': 2.85,
  'sha256': '7981744148cc25fb50002d5d886e731b417ee5a429ee5b13c2939b22432ca533'},
 {'threshold': 0.58,
  'zip_path': '/kaggle/working/runs/v4_di

In [31]:
RUN_DIR = Path('/kaggle/working')

In [32]:
results = submission_results + submission_results_ema

In [34]:
from pathlib import Path
import zipfile
import json

bundle_path = RUN_DIR / 'all_submissions_bundle.zip'

candidate_keys = ['zip_path', 'submission', 'zip']
zip_paths = []

for row in results:
    p = None
    for k in candidate_keys:
        if k in row and row[k]:
            p = row[k]
            break
    if p is None:
        continue
    p = Path(p)
    if p.exists():
        zip_paths.append(p)

print('found zip files:', len(zip_paths))
for p in zip_paths:
    print(p)



found zip files: 40
/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p50.zip
/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p52.zip
/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p54.zip
/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p56.zip
/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p58.zip
/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p60.zip
/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p62.zip
/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p64.zip
/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p66.zip
/kaggle/working/runs/v4_dinov2_cleaned/submissions/submission_v4_dinov2_cleaned_best_t_0p68.zip
/kaggle/working/runs

In [35]:
if bundle_path.exists():
    bundle_path.unlink()

with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    manifest = []
    for p in zip_paths:
        zf.write(p, arcname=p.name)
        manifest.append({
            'name': p.name,
            'path': str(p),
            'size_mb': round(p.stat().st_size / 1e6, 3),
        })
    zf.writestr('manifest.json', json.dumps(manifest, indent=2, ensure_ascii=False))

with zipfile.ZipFile(bundle_path, 'r') as zf:
    bad = zf.testzip()
    assert bad is None, bad

print('bundle created:', bundle_path)
print('size_mb:', round(bundle_path.stat().st_size / 1e6, 3))


bundle created: /kaggle/working/all_submissions_bundle.zip
size_mb: 84.283
